In [1]:
!pip install torch torchvision torchaudio transformers librosa soundfile scikit-learn pandas tqdm

In [5]:
import os
import random
import math
from pathlib import Path
from typing import List, Dict, Optional

import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

import torchaudio
from torchaudio import transforms as T
import librosa

from transformers import Wav2Vec2Model, Wav2Vec2FeatureExtractor, get_cosine_schedule_with_warmup
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight

# Global constants
DEFAULT_SAMPLE_RATE = 16000
MAX_SECONDS = 7
MAX_SAMPLES = DEFAULT_SAMPLE_RATE * MAX_SECONDS
NUM_CLASSES = 4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

In [ ]:
class RandomApply:
    def __init__(self, fn, p=0.5):
        self.fn = fn
        self.p = p
    def __call__(self, x):
        return self.fn(x) if random.random() < self.p else x

class AddGaussianNoise:
    def __init__(self, min_snr=5, max_snr=20):
        self.min_snr = min_snr
        self.max_snr = max_snr
    def __call__(self, audio: torch.Tensor):
        snr = random.uniform(self.min_snr, self.max_snr)
        rms = torch.sqrt(torch.mean(audio**2) + 1e-9)
        noise_rms = rms / (10**(snr/20))
        noise = torch.randn_like(audio) * noise_rms
        return audio + noise

class RandomGain:
    def __init__(self, min_db=-6, max_db=6):
        self.min_db = min_db
        self.max_db = max_db
    def __call__(self, audio: torch.Tensor):
        db = random.uniform(self.min_db, self.max_db)
        gain = 10 ** (db / 20)
        return audio * gain

class RandomSpeed:
    def __init__(self, min_rate=0.9, max_rate=1.1, sr=DEFAULT_SAMPLE_RATE):
        self.min_rate = min_rate
        self.max_rate = max_rate
        self.sr = sr
    def __call__(self, audio: torch.Tensor):
        rate = random.uniform(self.min_rate, self.max_rate)
        if abs(rate - 1.0) < 1e-3:
            return audio
        new_sr = int(self.sr * rate)
        resampled = torchaudio.functional.resample(audio, self.sr, new_sr)
        return torchaudio.functional.resample(resampled, new_sr, self.sr)

class SpecAugment:
    def __init__(self, freq_mask_param=15, time_mask_param=35, num_masks=2):
        self.freq_mask = T.FrequencyMasking(freq_mask_param)
        self.time_mask = T.TimeMasking(time_mask_param)
        self.num_masks = num_masks
    def __call__(self, spec: torch.Tensor):
        for _ in range(self.num_masks):
            spec = self.freq_mask(spec)
            spec = self.time_mask(spec)
        return spec

class PolarityInversion:
    """Flips the amplitude of the entire waveform."""
    def __call__(self, audio: torch.Tensor):
        return audio * -1.0

class TimeShift:
    """Shifts the audio in time by a random amount, wrapping the ends."""
    def __init__(self, max_shift_ms=100, sr=DEFAULT_SAMPLE_RATE):
        self.max_shift = int((max_shift_ms / 1000.0) * sr)
    
    def __call__(self, audio: torch.Tensor):
        if self.max_shift == 0:
            return audio
        shift = random.randint(-self.max_shift, self.max_shift)
        return torch.roll(audio, shifts=shift, dims=-1)


class RandomClipping:
    """Applies hard clipping to the audio signal."""
    def __init__(self, min_clip=0.5, max_clip=1.0):
        self.min_clip = min_clip
        self.max_clip = max_clip
        
    def __call__(self, audio: torch.Tensor):
        clip_val = random.uniform(self.min_clip, self.max_clip)
        return torch.clamp(audio, -clip_val, clip_val)

class RandomBandPassFilter:
    """Applies a random band-pass filter using SoX."""
    def __init__(self, sr=DEFAULT_SAMPLE_RATE, min_center_freq=400, max_center_freq=4000, min_q=0.5, max_q=1.5):
        self.sr = sr
        self.min_center_freq = min_center_freq
        self.max_center_freq = max_center_freq
        self.min_q = min_q
        self.max_q = max_q
        
    def __call__(self, audio: torch.Tensor):
        center_freq = random.uniform(self.min_center_freq, self.max_center_freq)
        q = random.uniform(self.min_q, self.max_q)
        
        try:

            effects = [
                ["bandpass", str(center_freq), f"{q}q"]
            ]
            augmented_audio, _ = torchaudio.sox_effects.apply_effects_tensor(
                audio, self.sr, effects
            )
            if augmented_audio.abs().max() < 1e-5:
                return audio
            return augmented_audio
        except Exception:
            return audio

In [ ]:
def _load_audio_for_precompute(
    path: Path, 
    sample_rate: int = DEFAULT_SAMPLE_RATE, 
    max_samples: int = MAX_SAMPLES
) -> torch.Tensor:
    """
    Helper function to load, resample, and pad/trim.
    This is the non-augmenting version of the original _load_audio.
    """
    wav, sr = torchaudio.load(str(path))
    if wav.shape[0] > 1:
        wav = torch.mean(wav, dim=0, keepdim=True)
    if sr != sample_rate:
        wav = torchaudio.functional.resample(wav, sr, sample_rate)
    
    if wav.shape[1] > max_samples:
        wav = wav[:, :max_samples]
    elif wav.shape[1] < max_samples:
        wav = F.pad(wav, (0, max_samples - wav.shape[1]))
    return wav

In [ ]:
class DeepfakeAudioDataset(Dataset):
    def __init__(
        self,
        df: pd.DataFrame,
        feature_dir: str,  
        audio_dir: str,    
        augment: bool = True,
    ):
        self.df = df.reset_index(drop=True)
        self.feature_dir = Path(feature_dir)
        self.audio_dir = Path(audio_dir)
        self.augment = augment

        self.spec_aug = SpecAugment(freq_mask_param=30, time_mask_param=70, num_masks=3)

        self.noise = RandomApply(AddGaussianNoise(min_snr=5, max_snr=20), p=0.4)
        self.gain = RandomApply(RandomGain(min_db=-6, max_db=6), p=0.3)
        self.polarity = RandomApply(PolarityInversion(), p=0.5)
        self.time_shift = RandomApply(TimeShift(max_shift_ms=150), p=0.3)
        self.clipping = RandomApply(RandomClipping(), p=0.2)
        self.filtering = RandomApply(RandomBandPassFilter(), p=0.3)


    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        filename = row['filename']
        # -------------------------
        
        label = int(row['target']) if 'target' in row else -1
        
        feature_path = self.feature_dir / f"{filename}.pt"
        original_audio_path = self.audio_dir / filename

        try:
            data = torch.load(feature_path, map_location='cpu')
            spec = data['spec'] 

            wav = _load_audio_for_precompute(original_audio_path) 
            
        except Exception as e:
            wav = torch.zeros(1, MAX_SAMPLES, dtype=torch.float) 
            spec = torch.zeros(2, 128, 313, dtype=torch.float) 
            label = 0 

        if self.augment:
            wav = self.noise(wav)
            wav = self.gain(wav)
            wav = self.polarity(wav)
            wav = self.time_shift(wav)
            wav = self.filtering(wav) 
            wav = self.clipping(wav)   
            
            spec = self.spec_aug(spec)
        
        wav = wav.squeeze(0) 

        return {
            'wav': wav,
            'spec': spec,
            'label': label,
            'Filename': filename 
        }

def collate_fn(batch: List[Dict], feature_extractor: Wav2Vec2FeatureExtractor):
    """This function remains identical to your original code."""
    wavs = [item['wav'] for item in batch]
    specs = [item['spec'] for item in batch]
    labels = torch.tensor([item['label'] for item in batch], dtype=torch.long)

    specs = torch.stack(specs, dim=0) 

    raw_list = [w.cpu().numpy() if isinstance(w, torch.Tensor) else w for w in wavs]
    inputs = feature_extractor(raw_list, sampling_rate=feature_extractor.sampling_rate, return_tensors='pt', padding=True)
    input_values = inputs['input_values']    # (B, L)
    attention_mask = inputs.get('attention_mask', None)

    return {
        'input_values': input_values,
        'attention_mask': attention_mask,
        'specs': specs,
        'labels': labels
    }

In [ ]:
import torchvision.models as models

def unfreeze_resnet_blocks(resnet_model: nn.Module, unfreeze_blocks: int):
    """
    Unfreeze the last `unfreeze_blocks` blocks among [layer1, layer2, layer3, layer4].
    If unfreeze_blocks=1 -> unfreeze layer4 only; 2-> layer3 & layer4; etc.
    """
    # first freeze all
    for p in resnet_model.parameters():
        p.requires_grad = False
    # map blocks
    layers = [resnet_model.layer1, resnet_model.layer2, resnet_model.layer3, resnet_model.layer4]
    if unfreeze_blocks <= 0:
        return
    unfreeze_blocks = min(unfreeze_blocks, 4)
    for layer in layers[-unfreeze_blocks:]:
        for p in layer.parameters():
            p.requires_grad = True

class SpectrogramNet(nn.Module):
    def __init__(self, in_ch=2, backbone_name='resnet50', resnet_unfreeze_blocks=3, pretrained=True):
        super().__init__()
        if backbone_name == 'resnet50':
            base = models.resnet50(pretrained=pretrained)
            self.feat_dim = 2048
        elif backbone_name == 'resnet34':
            base = models.resnet34(pretrained=pretrained)
            self.feat_dim = 512
        else:
            raise ValueError("backbone_name must be resnet34 or resnet50")
        
        # adapt first conv
        base.conv1 = nn.Conv2d(in_ch, 64, kernel_size=7, stride=2, padding=3, bias=False)
        # encoder until last conv
        self.encoder = nn.Sequential(*list(base.children())[:-2])
        self.pool = nn.AdaptiveAvgPool2d((1,1))
        
        # We REMOVED the self.fc projection layer as requested
        
        # freeze/unfreeze last blocks
        unfreeze_resnet_blocks(base, resnet_unfreeze_blocks)

    def forward(self, x):
        feat = self.encoder(x)
        pooled = self.pool(feat).view(feat.size(0), -1)
        return pooled 

class AudioTransformerBranch(nn.Module):
    def __init__(self, model_name='microsoft/wavlm-base-plus', unfreeze_last_n=6):
        super().__init__()
        self.wav2vec = Wav2Vec2Model.from_pretrained(model_name)
        
        # freeze all first
        for p in self.wav2vec.parameters():
            p.requires_grad = False
            
        # unfreeze last N transformer layers
        try:
            layers = self.wav2vec.encoder.layers
        except Exception:
            layers = self.wav2vec.encoder.layer
            
        total = len(layers)
        n = min(unfreeze_last_n, total)
        if n > 0:
            for layer in layers[-n:]:
                for p in layer.parameters():
                    p.requires_grad = True
                    
        self.hidden_size = self.wav2vec.config.hidden_size 
        

    def forward(self, input_values, attention_mask=None):
        out = self.wav2vec(input_values, attention_mask=attention_mask)
        last_hidden = out.last_hidden_state 
        
        if attention_mask is not None:
            padding_mask = self.wav2vec._get_feature_vector_attention_mask(last_hidden.shape[1], attention_mask)
            hidden_states = last_hidden.masked_fill(~padding_mask.bool().unsqueeze(-1), 0.0)
            sum_hidden = hidden_states.sum(dim=1)
            num_non_padded = padding_mask.sum(dim=1).unsqueeze(-1)
            emb = sum_hidden / (num_non_padded + 1e-9)
        else:
            emb = torch.mean(last_hidden, dim=1)
            
        return emb 

class FusionModel(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, wav2vec_model='microsoft/wavlm-base-plus', resnet_backbone='resnet50',
                 resnet_unfreeze_blocks=3, wav_unfreeze_last_n=6):
        super().__init__()
        
        # 1. Init branches without out_dim
        self.wav_branch = AudioTransformerBranch(model_name=wav2vec_model, unfreeze_last_n=wav_unfreeze_last_n)
        self.spec_branch = SpectrogramNet(in_ch=2, backbone_name=resnet_backbone, resnet_unfreeze_blocks=resnet_unfreeze_blocks)
        
        # 2. Get their native output dimensions
        wav_out_dim = self.wav_branch.hidden_size   
        spec_out_dim = self.spec_branch.feat_dim  
        
        concat_dim = wav_out_dim + spec_out_dim 
        print(f"Initializing FusionModel head with input dim: {concat_dim}")
        
        self.head = nn.Sequential(
            nn.Linear(concat_dim, 1024), # Project from 2816 -> 1024
            nn.ReLU(),
            nn.BatchNorm1d(1024),
            nn.Dropout(0.5),
            nn.Linear(1024, 512),        
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)  # Final classification
        )

    def forward(self, input_values, specs, attention_mask=None):
        wav_emb = self.wav_branch(input_values, attention_mask=attention_mask)   
        spec_emb = self.spec_branch(specs)                                   
        
        x = torch.cat([wav_emb, spec_emb], dim=1)                              
        logits = self.head(x)
        return logits

In [ ]:
from torch.cuda.amp import autocast, GradScaler


def get_class_weights(train_df):
    labels = train_df['target'].values
    classes = np.unique(labels)
    weights = compute_class_weight(class_weight='balanced', classes=classes, y=labels)
    full_weights = np.ones(NUM_CLASSES, dtype=float)
    for c, w in zip(classes, weights):
        full_weights[int(c)] = w
    print("Class weights:", full_weights)
    return torch.tensor(full_weights, dtype=torch.float)

def train_one_epoch(model, loader, optimizer, scheduler, device, scaler, criterion, epoch, print_every=50):
    model.train()
    running_loss = 0.0
    all_preds, all_labels = [], []
    pbar = tqdm(enumerate(loader), total=len(loader), desc=f"Train Epoch {epoch}")
    for i, batch in pbar:
        input_values = batch['input_values'].to(device)
        attention_mask = batch['attention_mask']
        if attention_mask is not None:
            attention_mask = attention_mask.to(device)
        specs = batch['specs'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()
        
        with torch.amp.autocast(device_type=DEVICE.type, dtype=torch.float16):
            logits = model(input_values, specs, attention_mask)
            loss = criterion(logits, labels)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        if scheduler is not None:
            scheduler.step()

        running_loss += loss.item() * labels.size(0)
        preds = torch.argmax(logits, dim=1).detach().cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.detach().cpu().numpy())

        if (i+1) % print_every == 0:
            cur_f1 = f1_score(all_labels, all_preds, average='macro') if len(all_labels) > 0 else 0.0
            pbar.set_postfix({'loss': loss.item(), 'f1': f"{cur_f1:.4f}"})

    avg_loss = running_loss / len(loader.dataset)
    macro_f1 = f1_score(all_labels, all_preds, average='macro')
    return avg_loss, macro_f1

def validate_one_epoch(model, loader, device, criterion):
    model.eval()
    running_loss = 0.0
    all_preds, all_labels = [], []
    pbar = tqdm(loader, total=len(loader), desc="Validate ")
    with torch.no_grad():
        for batch in pbar:
            input_values = batch['input_values'].to(device)
            attention_mask = batch['attention_mask']
            if attention_mask is not None:
                attention_mask = attention_mask.to(device)
            specs = batch['specs'].to(device)
            labels = batch['labels'].to(device)

            with torch.amp.autocast(device_type=DEVICE.type, dtype=torch.float16):
                logits = model(input_values, specs, attention_mask)
                loss = criterion(logits, labels)

            running_loss += loss.item() * labels.size(0)
            preds = torch.argmax(logits, dim=1).detach().cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.detach().cpu().numpy())

            cur_f1 = f1_score(all_labels, all_preds, average='macro') if len(all_labels) > 0 else 0.0
            pbar.set_postfix({'loss': loss.item(), 'f1': f"{cur_f1:.4f}"})

    avg_loss = running_loss / len(loader.dataset)
    macro_f1 = f1_score(all_labels, all_preds, average='macro')
    return avg_loss, macro_f1

In [ ]:

def _compute_cqt_for_precompute(
    audio_np: np.ndarray, 
    sample_rate: int = DEFAULT_SAMPLE_RATE, 
    cqt_bins: int = 84,
    n_mels: int = 128
) -> np.ndarray:
    """Helper function to compute CQT."""
    try:
        cqt = librosa.cqt(audio_np, sr=sample_rate, hop_length=256, n_bins=cqt_bins)
        cqt_mag = np.abs(cqt)
        cqt_db = librosa.amplitude_to_db(cqt_mag, ref=np.max)
        return cqt_db
    except Exception:
        mel = librosa.feature.melspectrogram(audio_np, sr=sample_rate, n_fft=1024, hop_length=256, n_mels=n_mels)
        mel_db = librosa.power_to_db(mel, ref=np.max)
        return mel_db

def precompute_features(
    df: pd.DataFrame, 
    audio_dir: str, 
    output_dir: str, 
    n_mels: int = 128, 
    cqt_bins: int = 84
):
    """
    Main precomputation function.
    Iterates over the dataframe, processes each file, and saves *only*
    the 'spec' dictionary to the output_dir.
    """
    audio_dir = Path(audio_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    mel_t = T.MelSpectrogram(sample_rate=DEFAULT_SAMPLE_RATE, n_fft=1024, hop_length=256, n_mels=n_mels)
    to_db = T.AmplitudeToDB()

    all_Files = df['filename'].unique() 
    print(f"Starting precomputation for {len(all_Files)} unique files...")

    for filename in tqdm(all_Files, desc="Precomputing features"): 
        out_path = output_dir / f"{filename}.pt" 
    # -------------------------
    
        if out_path.exists():
            continue
            
        path = audio_dir / filename 
        try:
            # 1. Load and process raw wav
            wav = _load_audio_for_precompute(path) 
            wav_np = wav.squeeze(0).cpu().numpy()

            # 2. Compute Mel Spectrogram
            mel = mel_t(wav)  
            mel = to_db(mel)
            mel = (mel - mel.mean()) / (mel.std() + 1e-6)

            # 3. Compute CQT
            cqt_db = _compute_cqt_for_precompute(wav_np, n_mels=n_mels, cqt_bins=cqt_bins)
            cqt_t = torch.tensor(cqt_db, dtype=torch.float).unsqueeze(0) 

            # 4. Align CQT and Mel
            target_time = mel.shape[-1]
            if cqt_t.shape[2] < target_time:
                pad = target_time - cqt_t.shape[2]
                cqt_t = F.pad(cqt_t, (0, pad))
            elif cqt_t.shape[2] > target_time:
                cqt_t = cqt_t[:, :, :target_time]
            
            # 5. Resample CQT bins to match Mel bins
            if cqt_t.shape[1] != mel.shape[1]:
                cqt_t = F.interpolate(cqt_t.unsqueeze(0), size=(mel.shape[1], mel.shape[2]), mode='bilinear', align_corners=False).squeeze(0)
            
            cqt_t = (cqt_t - cqt_t.mean()) / (cqt_t.std() + 1e-6)

            # 6. Stack channels
            spec = torch.cat([mel, cqt_t], dim=0) 
            
            # 7. Save *ONLY* the processed spec
            data_to_save = {
                'spec': spec
            }
            torch.save(data_to_save, out_path)

        except Exception as e:

            print(f"Failed to process {filename}: {e}") \
            # -------------------------

    print("Precomputation finished.")

In [ ]:
def run_training(
    train_csv: str,
    audio_dir: str,  
    wav2vec_model: str = 'microsoft/wavlm-base-plus',
    backbone: str = 'resnet50',
    resnet_unfreeze_blocks: int = 2,
    wav_unfreeze_last_n: int = 2,
    epochs: int = 70,
    batch_size: int = 16,
    lr: float = 6e-5,
    weight_decay: float = 0.01,   
    label_smoothing: float = 0.1,  
    warmup_ratio: float = 0.05,
    patience: int = 12,
    num_workers: int = 4,
    save_path: str = '/kaggle/working/model.pth'
):
    set_seed(42)
    df = pd.read_csv(train_csv)
    
    feature_dir = "/kaggle/working/precomputed_features/"
    precompute_features(df, audio_dir, feature_dir)

    df = df.sample(frac=1, random_state=42).reset_index(drop=True)
    split = int(0.9 * len(df))
    train_df = df[:split].reset_index(drop=True)
    val_df = df[split:].reset_index(drop=True)
    print(f"Train samples: {len(train_df)}, Val samples: {len(val_df)}")

    feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(wav2vec_model, sampling_rate=DEFAULT_SAMPLE_RATE)

    train_ds = DeepfakeAudioDataset(train_df, feature_dir, audio_dir, augment=True)
    val_ds = DeepfakeAudioDataset(val_df, feature_dir, audio_dir, augment=False)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers,
                              collate_fn=lambda b: collate_fn(b, feature_extractor), pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size*2, shuffle=False, num_workers=num_workers,
                            collate_fn=lambda b: collate_fn(b, feature_extractor), pin_memory=True)

    model = FusionModel(num_classes=NUM_CLASSES, wav2vec_model=wav2vec_model,
                        resnet_backbone=backbone, resnet_unfreeze_blocks=resnet_unfreeze_blocks,
                        wav_unfreeze_last_n=wav_unfreeze_last_n).to(DEVICE)

    class_weights = get_class_weights(train_df).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=label_smoothing)
    print(f"Using CrossEntropyLoss with label smoothing={label_smoothing}")

    trainable_params = [p for p in model.parameters() if p.requires_grad]
    print("Trainable params count:", sum(p.numel() for p in trainable_params))
    optimizer = AdamW(trainable_params, lr=lr, weight_decay=weight_decay)
    print(f"Using AdamW optimizer with weight_decay={weight_decay}")

    total_steps = len(train_loader) * epochs
    warmup_steps = max(1, int(total_steps * warmup_ratio))
    scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)

    scaler = torch.amp.GradScaler(enabled=(DEVICE.type == 'cuda'))
    
    
    best_f1 = 0.0
    train_f1_at_best_val = 0.0 
    epochs_since_last_save = 0

    for epoch in range(1, epochs+1):
        train_loss, train_f1 = train_one_epoch(model, train_loader, optimizer, scheduler, DEVICE, scaler, criterion, epoch)
        val_loss, val_f1 = validate_one_epoch(model, val_loader, DEVICE, criterion)
        print(f"Epoch {epoch}/{epochs} -> TrainLoss: {train_loss:.4f} TrainF1: {train_f1:.4f} | ValLoss: {val_loss:.4f} ValF1: {val_f1:.4f}")

        
        a_model_was_saved = False 

        is_new_best_val = val_f1 > best_f1

        is_new_best_train = (val_f1 >= best_f1) and (train_f1 > train_f1_at_best_val)


        if is_new_best_val:
            print(f"--- New Best Validation F1 ---")
            best_f1 = val_f1 
            train_f1_at_best_val = train_f1
            a_model_was_saved = True 
            
            torch.save({
                'model_state': model.state_dict(),
                'optimizer_state': optimizer.state_dict(),
                'scheduler_state': scheduler.state_dict(),
                'epoch': epoch,
                'val_f1': val_f1,
                'train_f1': train_f1 
            }, save_path)
            print(f"Saved BEST VALIDATION model to {save_path} (ValF1={val_f1:.4f}, TrainF1={train_f1:.4f})")

        elif is_new_best_train:
            print(f"--- New Best Conditional Train F1 ---")
            train_f1_at_best_val = train_f1 
            a_model_was_saved = True 
            
            p = Path(save_path)
            best_train_save_path = p.with_name(f"{p.stem}_best_train{p.suffix}")
            torch.save({
                'model_state': model.state_dict(),
                'optimizer_state': optimizer.state_dict(),
                'scheduler_state': scheduler.state_dict(),
                'epoch': epoch,
                'val_f1': val_f1,
                'train_f1': train_f1 
            }, best_train_save_path)
            print(f"Saved BEST TRAIN model to {best_train_save_path} (ValF1={val_f1:.4f}, TrainF1={train_f1:.4f})")

        # --- Early Stopping Logic ---
        
        if a_model_was_saved:
            print(f"Model saved, resetting early stopping counter to 0.")
            epochs_since_last_save = 0 # Reset counter
        else:
            epochs_since_last_save += 1 # Increment counter
            print(f"No improvement. Early stopping counter: {epochs_since_last_save}/{patience}")

        if epochs_since_last_save >= patience:
            print("Early stopping triggered.")
            break
            

    print(f"Training finished. Best Val F1: {best_f1:.4f}")
    return model

In [ ]:
train_csv = "/kaggle/input/comsys-hackathon-6-task-b-2-0/train_new.csv"  
audio_dir = "/kaggle/input/comsys-hackathon-6-task-b-2-0/ComsysTaskB/train"         


model = run_training(
    train_csv=train_csv,
    audio_dir=audio_dir,
    wav2vec_model='microsoft/wavlm-base-plus',  
    backbone='resnet50',                      
    resnet_unfreeze_blocks=2,                  
    wav_unfreeze_last_n=2,                        
    epochs=70,
    batch_size=16,                            
    lr=6e-5,                                 
    label_smoothing=0.1,                     
    warmup_ratio=0.1,
    patience=25,
    num_workers=4,
    save_path='/kaggle/working/best_fusion_model.pth'
)

Starting precomputation for 14400 unique files...


Precomputing features: 100%|██████████| 14400/14400 [00:00<00:00, 123707.54it/s]

Precomputation finished.
Train samples: 12960, Val samples: 1440



You are using a model of type wavlm to instantiate a model of type wav2vec2. This is not supported for all configurations of models and can yield errors.
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Initializing FusionModel head with input dim: 2816
Class weights: [1.48965517 0.75454122 1.49377593 0.74965294]
Using CrossEntropyLoss with label smoothing=0.1
Trainable params count: 39653380
Using AdamW optimizer with weight_decay=0.01


...

Epoch 68/70 -> TrainLoss: 0.3992 TrainF1: 0.9999 | ValLoss: 0.3905 ValF1: 1.0000
No improvement. Early stopping counter: 7/25



Validate : 100%|██████████| 45/45 [00:25<00:00,  1.74it/s, loss=0.421, f1=1.0000]

Epoch 69/70 -> TrainLoss: 0.3995 TrainF1: 0.9999 | ValLoss: 0.3907 ValF1: 1.0000
No improvement. Early stopping counter: 8/25



Validate : 100%|██████████| 45/45 [00:25<00:00,  1.75it/s, loss=0.42, f1=1.0000] 

Epoch 70/70 -> TrainLoss: 0.3991 TrainF1: 0.9999 | ValLoss: 0.3902 ValF1: 1.0000
No improvement. Early stopping counter: 9/25
Training finished. Best Val F1: 1.0000
